### 导入数据集

In [88]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

In [89]:
iris = load_iris()
X = iris.data
y = iris.target

In [90]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=233, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((105, 4), (45, 4), (105,), (45,))

### KNN超参数搜索

In [91]:
from sklearn.neighbors import KNeighborsClassifier

In [92]:
best_score = -1  # 保存最优结果
best_n = -1
best_weight = ""
best_p = -1
for n in range(1, 20):  # 遍历不同的超参数组合
    for weight in ["uniform", "distance"]:
        for p in range(1, 7):
            neigh = KNeighborsClassifier(  # 创建 KNN 分类模型
                n_neighbors=n,   # 邻居数量
                weights=weight,  # 权重方式
                p=p)             # 距离参数
            neigh.fit(X_train, y_train)  # 训练模型
            score = neigh.score(X_test, y_test)  # 计算测试集准确率
            if score > best_score:  # 更新最优参数
                best_score = score
                best_n = n
                best_weight = weight
                best_p = p
print("n_neighbors:", best_n)  # 输出最优参数和得分
print("weights:", best_weight)
print("p:", best_p)
print("score:", best_score)

n_neighbors: 5
weights: uniform
p: 2
score: 1.0


### sklearn中的KNN超参数搜索

In [93]:
from sklearn.model_selection import GridSearchCV

In [94]:
params = {                                     # 设置超参数搜索范围
    "n_neighbors": [n for n in range(1, 20)],  # 邻居数量
    "weights": ["uniform", "distance"],        # 权重方式
    "p": [p for p in range(1, 7)]}             # 距离参数

In [95]:
grid = GridSearchCV(                   # 创建网格搜索
    estimator=KNeighborsClassifier(),  # KNN模型
    param_grid=params,                 # 参数组合
    n_jobs=-1)                         # 使用全部CPU核心

In [96]:
grid.fit(X_train, y_train);  # 搜索最优参数

In [97]:
grid.best_params_  # 查看最优参数

{'n_neighbors': 9, 'p': 2, 'weights': 'uniform'}

In [98]:
grid.best_score_  # 查看最佳交叉验证得分

np.float64(0.961904761904762)

### 交叉验证

In [99]:
from sklearn.model_selection import cross_val_score

In [100]:
neigh = KNeighborsClassifier()  # 创建 KNN 模型
cv_scores = cross_val_score(neigh, X_train, y_train, cv=5)  # 进行 5 折交叉验证
print(cv_scores)  # 输出每一折的得分

[0.95238095 1.         0.95238095 0.85714286 1.        ]


In [101]:
best_score = -1  # 初始化最优结果
best_n = -1
best_weight = ""
best_p = -1
best_cv_scores = None
for n in range(1, 20):  # 遍历超参数组合
    for weight in ["uniform", "distance"]:
        for p in range(1, 7):
            neigh = KNeighborsClassifier(  # 创建 KNN 模型
                n_neighbors=n,
                weights=weight,
                p=p)
            cv_scores = cross_val_score(neigh, X_train, y_train, cv=5)  # 进行 5 折交叉验证
            score = np.mean(cv_scores)  # 计算平均得分
            if score > best_score:  # 更新最优参数
                best_score = score
                best_n = n
                best_weight = weight
                best_p = p
                best_cv_scores = cv_scores
print("n_neighbors:", best_n)  # 输出最优结果
print("weights:", best_weight)
print("p:", best_p)
print("score:", best_score)
print("best_cv_scores:", best_cv_scores)

n_neighbors: 9
weights: uniform
p: 2
score: 0.961904761904762
best_cv_scores: [1.         1.         0.95238095 0.85714286 1.        ]
